# Find an Antminer build and its installation instructions

An offline example for the VNISH Verified Firmware Catalog

Select the exact model and control board, inspect one default build in the saved catalog, then follow the published instruction link for VNISH GLOBAL, ROI ASIC or VNISH Ninja

The example reads repository JSON files only. It makes no network requests, downloads no firmware and does not connect to a miner

The recorded output belongs to the catalog snapshot shown below. A saved default is not a live check of today's release. Confirm the physical model, board and current stock firmware in the linked instructions before installation

[English guide](catalog-selection.md) | [Русская инструкция](catalog-selection.ru.md)


## 1. Read the saved catalog

Open this notebook from the full repository checkout, either at its root or in `examples/`. Python's standard library is sufficient for the code cells

The catalog digest checks the local catalog file. Published firmware hashes are read as metadata; no firmware binary is verified by this notebook


In [1]:
from pathlib import Path
import hashlib
import json
import re

# Run from the repository root or its examples directory
ROOT = Path.cwd()
if not (ROOT / "data/current/catalog.json").is_file():
    ROOT = ROOT.parent
DATA = ROOT / "data/current"
payload = (DATA / "catalog.json").read_bytes()
expected = (DATA / "DIGEST").read_text().split()[0]
actual = hashlib.sha256(payload).hexdigest()
if actual != expected:
    raise ValueError("Catalog differs from its saved DIGEST")
catalog = json.loads(payload)
sources = {
    host: json.loads((DATA / "metadata-sources" / (host + ".json")).read_text())
    for host in ("vnish.global", "roiasic.com", "vnish.ninja")
}
print("Catalog snapshot:", catalog["release"], "updated", catalog["updated"])
print("Models:", len(catalog["models"]), "build records:", len(catalog["builds"]))
print("Default records in this snapshot:", sum(b["is_default"] is True for b in catalog["builds"]))
print("Catalog SHA-256:", actual)


Catalog snapshot: catalog-2026-09-24 updated 2026-09-24
Models: 47 build records: 224
Default records in this snapshot: 76
Catalog SHA-256: b4aa8dfe50dbe49235f971fcd0f96eaa1df10002831a86526ab693743292b0fb


## 2. Match exact identifiers

The choice must match one model, one control-board code, one package method and one default build. The three saved source catalogs must agree on the file, version, size and published hash

`nand` identifies the package method in this dataset. It does not establish that a device's present stock firmware permits a particular installation procedure


In [2]:
def one(items, message):
    if len(items) != 1:
        raise ValueError(message + " (matches: " + str(len(items)) + ")")
    return items[0]


def select_default(model_id, board_code, install_method="nand"):
    """Return one snapshot default and its published instruction links"""
    for value in (model_id, board_code, install_method):
        if not isinstance(value, str) or not value or not re.fullmatch(r"[a-z0-9-]+", value):
            raise ValueError("Use explicit catalog IDs, for example l9, aml, nand")
    model = one([m for m in catalog["models"] if m["model_id"] == model_id], "Unknown or duplicate model")
    build = one([
        b for b in catalog["builds"]
        if b["model_id"] == model_id
        and b["control_board_code"] == board_code
        and b["install_method"] == install_method
        and b["is_default"] is True
    ], "Expected exactly one default for this model, board and method")
    if build["build_id"] not in model["default_build_ids"]:
        raise ValueError("Model and build disagree about the default")
    if not re.fullmatch(r"[a-z0-9-]+", build["route_id"]):
        raise ValueError("Unexpected route ID")
    links = {}
    for host, source in sources.items():
        if host == "vnish.global":
            raw = one([b for b in source["builds"] if b["id"] == build["build_id"]], "Missing or duplicate Global source record")
            if raw["is_default"] is not True or source["default_version"] != build["firmware_version"]:
                raise ValueError("Global source default disagrees with catalog")
            identity = (raw["model_id"], raw["board_platform"]["code"], raw["install_method"])
            file_name, size = raw["file_name"], raw["size_bytes"]
        else:
            raw_model = one([m for m in source["models"] if m["model"] == model_id], "Missing or duplicate site model")
            raw = raw_model["boards"].get(board_code)
            if raw is None:
                raise ValueError("Board absent from site source")
            identity = (raw_model["model"], board_code, raw["install_type"])
            file_name, size = raw["file"], raw["size"]
        if identity != (model_id, board_code, install_method):
            raise ValueError("Source hardware identity disagrees")
        if (raw["version"], file_name, size, raw["sha256"]) != (
            build["firmware_version"], build["file_name"], build["size_bytes"], build["sha256"]
        ):
            raise ValueError("Source metadata disagrees with catalog")
        guide = raw["installation_guide"]
        # Use the recorded guide; reject unexpected or archive routes
        if guide != "/install/" + build["route_id"] + "/":
            raise ValueError("Source has no matching current instruction route")
        links[host] = "https://" + host + guide
    return {
        "snapshot": catalog["release"], "updated": catalog["updated"],
        "build_id": build["build_id"], "model": model["name"],
        "board": build["control_board"], "method": build["install_method"],
        "version": build["firmware_version"], "file_name": build["file_name"],
        "published_sha256": build["sha256"], "size_bytes": build["size_bytes"],
        "instructions": links,
    }


def show(result):
    print(json.dumps(result, indent=2, ensure_ascii=False))


## 3. Example: Antminer L9 with Amlogic

Use the model IDs listed in `catalog["models"]`, not a broad family name or a guessed board. The instruction URLs below come from the saved source catalogs and stay on each brand's own domain


In [3]:
# Change these only after identifying the exact model and control board
MODEL_ID = "l9"
BOARD_CODE = "aml"
show(select_default(MODEL_ID, BOARD_CODE))


{
  "snapshot": "catalog-2026-09-24",
  "updated": "2026-09-24",
  "build_id": "l9-aml-nand-v1.3.6",
  "model": "Antminer L9",
  "board": "AML",
  "method": "nand",
  "version": "1.3.6",
  "file_name": "vnish-l9-aml-nand-v1.3.6.tar.gz",
  "published_sha256": "49b01f9326b27cb1fd355b9be998f5bddca8b7f06c19286578612c714ef837ff",
  "size_bytes": 21566744,
  "instructions": {
    "vnish.global": "https://vnish.global/install/l9-aml-nand/",
    "roiasic.com": "https://roiasic.com/install/l9-aml-nand/",
    "vnish.ninja": "https://vnish.ninja/install/l9-aml-nand/"
  }
}


## 4. Check two common ambiguities

L9 AML and L9 CV require separate records. Antminer S19 and Antminer S19 (126) are also separate model IDs


In [4]:
# Same model, different board: a different build and instruction route
aml = select_default("l9", "aml")
cv = select_default("l9", "cv")
print("AML:", aml["build_id"])
print("CV:", cv["build_id"])
print("Different published hashes:", aml["published_sha256"] != cv["published_sha256"])

# Similar names are separate model IDs, not aliases
print("S19:", select_default("s19", "xil")["build_id"])
print("S19 (126):", select_default("s19-126", "xil")["build_id"])


AML: l9-aml-nand-v1.3.6
CV: l9-cv-nand-v1.3.6
Different published hashes: True
S19: s19-xil-nand-v1.3.6
S19 (126): s19-126-xil-nand-v1.3.6


## 5. Stop when information is missing

This example does not silently substitute another board, model or archived version. `routes.csv` lists snapshot defaults, while `builds.csv` and `catalog.json` also preserve previous releases. Matching only a route ID can therefore produce multiple versions


In [5]:
# A missing board or unsupported combination stops without guessing
for model_id, board in [("l9", ""), ("l9", "bb"), ("s99", "aml")]:
    try:
        select_default(model_id, board)
    except ValueError as error:
        print(repr((model_id, board)), "STOP:", error)

# route_id alone spans releases; this is why the default flag matters
versions = sorted({
    b["firmware_version"] for b in catalog["builds"]
    if b["route_id"] == "l9-aml-nand"
})
print("Versions with the same route ID:", versions)


('l9', '') STOP: Use explicit catalog IDs, for example l9, aml, nand
('l9', 'bb') STOP: Expected exactly one default for this model, board and method (matches: 0)
('s99', 'aml') STOP: Unknown or duplicate model (matches: 0)
Versions with the same route ID: ['1.3.4', '1.3.5', '1.3.6']


## Continue on the selected website

Open the matching instruction URL from the result, confirm the hardware and stock firmware, then use that site's current guide. For an archived release, use its published archive instructions; do not reuse this default-only example as an archive installer

Catalog data: [ODC-By-1.0](../LICENSE-DATA.txt). Documentation: [CC BY 4.0](../LICENSE-DOCS.txt). Attribution: VNISH GLOBAL, ROI ASIC and VNISH Ninja. Firmware binaries and trademarks retain their existing rights
